In [1]:
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array

In [2]:
# Load trajectories (.xtc) and topology file (.tpr)
trajfiles2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1_f.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep2_f.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep3_f.xtc",
            )
topfile2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1.tpr")
trajfiles3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1_f.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep2_f.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep3_f.xtc",
            )
topfile3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1.tpr")
bdn1 = mda.Universe(topfile2, trajfiles2[0])
bdn2 = mda.Universe(topfile2, trajfiles2[1])
bdn3 = mda.Universe(topfile2, trajfiles2[2])
bdn = mda.Universe(topfile2, trajfiles2)

bdq1 = mda.Universe(topfile3, trajfiles3[0])
bdq2 = mda.Universe(topfile3, trajfiles3[1])
bdq3 = mda.Universe(topfile3, trajfiles3[2])
bdq = mda.Universe(topfile3, trajfiles3)

In [3]:
import numpy as np
from MDAnalysis.analysis.distances import distance_array

def min_distance_timeseries(u, protein_resid, ligand_resid):

    protein = u.select_atoms(
        f"protein and resid {protein_resid} and not name H*"
    )

    ligand = u.select_atoms(
        f"resid {ligand_resid} and not name H*"
    )

    distances = []

    for ts in u.trajectory:

        d = distance_array(
            protein.positions,
            ligand.positions,
            box=u.dimensions
        )

        distances.append(np.min(d))

    return np.array(distances)

In [4]:
import os
import pandas as pd


def run_distance_analysis(
    systems,
    sites,
    outdir
):

    os.makedirs(outdir, exist_ok=True)

    for system_name, replicas in systems.items():

        system_dir = os.path.join(outdir, system_name)
        os.makedirs(system_dir, exist_ok=True)

        for site_name, (protein_resid, ligand_resid) in sites.items():

            print(
                f"{system_name} | {site_name}: "
                f"prot {protein_resid} ↔ lig {ligand_resid}"
            )

            site_dir = os.path.join(system_dir, site_name)
            os.makedirs(site_dir, exist_ok=True)

            all_replicas = []

            for rep_idx, u in enumerate(replicas, start=1):

                dist = min_distance_timeseries(
                    u,
                    protein_resid,
                    ligand_resid
                )

                df = pd.DataFrame({

                    "System": system_name,
                    "ProteinResid": protein_resid,
                    "LigandResid": ligand_resid,
                    "Replica": rep_idx,
                    "Frame": range(len(dist)),
                    "MinDistance": dist

                })

                all_replicas.append(df)

            combined = pd.concat(all_replicas, ignore_index=True)

            outpath = os.path.join(
                site_dir,
                "min_distance_timeseries.csv"
            )

            combined.to_csv(outpath, index=False)

            print(f"Saved -> {outpath}")

In [5]:
systems = {
    "BDN": [bdn1, bdn2, bdn3],
    "BDQ": [bdq1, bdq2, bdq3]
}

glu61_sites = {
    "laggging_1": [160, 1064],
    "lagging_2": [162, 1064],
    "lagging_3": [163, 1064],
    "leading_1": [207, 1065],
    "leading_2": [208, 1065],
    "leading_3": [211, 1065]
}

In [6]:
run_distance_analysis(
    systems=systems,
    sites=glu61_sites,
    outdir="/scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading"
)

BDN | laggging_1: prot 160 ↔ lig 1064


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDN/laggging_1/min_distance_timeseries.csv
BDN | lagging_2: prot 162 ↔ lig 1064


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDN/lagging_2/min_distance_timeseries.csv
BDN | lagging_3: prot 163 ↔ lig 1064


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDN/lagging_3/min_distance_timeseries.csv
BDN | leading_1: prot 207 ↔ lig 1065


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDN/leading_1/min_distance_timeseries.csv
BDN | leading_2: prot 208 ↔ lig 1065


...

Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDQ/lagging_2/min_distance_timeseries.csv
BDQ | lagging_3: prot 163 ↔ lig 1064


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDQ/lagging_3/min_distance_timeseries.csv
BDQ | leading_1: prot 207 ↔ lig 1065


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDQ/leading_1/min_distance_timeseries.csv
BDQ | leading_2: prot 208 ↔ lig 1065


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDQ/leading_2/min_distance_timeseries.csv
BDQ | leading_3: prot 211 ↔ lig 1065


Saved -> /scratch/gent/vo/000/gvo00003/vsc48847/distance_analysis/lagging_leading/BDQ/leading_3/min_distance_timeseries.csv
